In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import nibabel as nb
from scipy.ndimage import gaussian_filter
from scipy.ndimage import distance_transform_edt
from glob import glob
import tifffile as tiff
import sys
import os
from glob import glob
sys.path.append('../')
from scipy import ndimage as ndi
from skimage.measure import block_reduce
from pathlib import Path

import re
import pandas as pd

# from slice_structure_identification_functions import compute_signed_distance_weight as compute_signed_distance_weight
# from slice_structure_identification_functions import compute_signed_distance_weight_filled as compute_signed_distance_weight_filled

import importlib


import slice_registration_functions
importlib.reload(slice_registration_functions)
from slice_registration_functions import apply_coordinate_mapping_2d, downsample_image, build_centroid_image

In [ ]:
##Mute the "LoopExit" error from concurrent.futures, which i think we can safely ignore.

import logging

try:
    from gevent.exceptions import LoopExit
except Exception:
    LoopExit = None

_concurrent_futures_logger = logging.getLogger("concurrent.futures")

class _IgnoreLoopExitFilter(logging.Filter):
    def filter(self, record):
        if LoopExit is None or not record.exc_info:
            return True
        return not isinstance(record.exc_info[1], LoopExit)

_loop_exit_filter = _IgnoreLoopExitFilter()
_concurrent_futures_logger.addFilter(_loop_exit_filter)

In [ ]:
##This unmutes the filter from the cell above this. Keeping commented out for now so if we run all it stays muted. Hacky but whatever

# if "_concurrent_futures_logger" in globals() and "_loop_exit_filter" in globals():
#     try:
#         _concurrent_futures_logger.removeFilter(_loop_exit_filter)
#     except Exception:
#         pass

In [ ]:
# GLOBALS

mask_dir = Path('/data/neuralabc/johmat/phase_ml/processing/full_slide_masks/2026_05_14/2026_05_14_pct_hi99.5_lo95')
root_dir = Path('/tmp/zefir_sliceReg_optimized_v2_rescale_5_renamed_copies/')


input_res = 0.345
target_res = 50.0
rescale = target_res / input_res
rescale = int(round(rescale))

In [ ]:
# Parse mask files: 2026_05_14_pct_hi99.5_lo95_realID_195-1_zefirID_0239_Image_94_01_mask_pyr_stable.ome.tif
mask_pattern = re.compile(r'(?:.*_)?realID_[\d-]+_zefirID_(\d+)_(Image_[\d_]+)_mask.*\.ome\.tif$')
mask_records = {}
for f in sorted(mask_dir.glob('*realID_*_mask*.ome.tif')):
    m = mask_pattern.match(f.name)
    if m:
        zefir_id = f'zefir_{m.group(1)}'
        image_id = m.group(2)
        mask_records[zefir_id] = image_id

# Parse reg files: 2026_05_14_pct_hi99.5_lo95_realID_195-1_zefirID_0239_Image_21_cellCount_29_downsample_10p002um_pix.nii.gz
reg_pattern = re.compile(r'(?:.*_)?realID_[\d-]+_zefirID_(\d+)_(Image_[\d_]+)_cellCount_.*\.nii\.gz$')
reg_records = {}
for f in sorted(root_dir.glob('*realID_*_cellCount_*.nii.gz')):
    m = reg_pattern.match(f.name)
    if m:
        zefir_id = f'zefir_{m.group(1)}'
        image_id = m.group(2)
        reg_records[zefir_id] = image_id

# Build paired table on shared zefir IDs (unchanged)
all_zefir_ids = sorted(set(mask_records) | set(reg_records))
rows = []
for zid in all_zefir_ids:
    mask_img = mask_records.get(zid)
    reg_img  = reg_records.get(zid)
    match    = (mask_img == reg_img) if (mask_img and reg_img) else None
    rows.append({
        'zefir_id':      zid,
        'mask_image_id': mask_img,
        'reg_image_id':  reg_img,
        'match':         match,
    })

df = pd.DataFrame(rows)
mismatches = df[df['match'] == False]
print(f"Total zefir IDs: {len(df)}")
print(f"  mask only:  {df['reg_image_id'].isna().sum()}")
print(f"  reg only:   {df['mask_image_id'].isna().sum()}")
print(f"  matched:    {(df['match'] == True).sum()}")
print(f"  MISMATCHED: {len(mismatches)}")
print()
if not mismatches.empty:
    print("Mismatches:")
    print(mismatches.to_string(index=False))

In [ ]:
mask_ids = set(mask_records.keys())
reg_ids  = set(reg_records.keys())

mask_only = sorted(mask_ids - reg_ids)
reg_only  = sorted(reg_ids - mask_ids)

print(f"In mask but not reg ({len(mask_only)}):", mask_only)
print()
print(f"In reg but not mask ({len(reg_only)}):", reg_only[:20], '...' if len(reg_only) > 20 else '')

# Also check for gaps in the numeric sequence within each set
def find_gaps(id_set):
    nums = sorted(int(re.search(r'\d+', x).group()) for x in id_set)
    return [n for a, b in zip(nums, nums[1:]) for n in range(a+1, b) if b - a > 1]

print()
print("Numeric gaps in mask IDs:", find_gaps(mask_ids))
print("Numeric gaps in reg IDs:",  find_gaps(reg_ids))

In [ ]:
# Look at the full picture around the flip point, including matched rows
transition_ids = [f'zefir_{i:04d}' for i in range(100, 300)]
print("Full picture around the flip (0199-0219):")
print(df[df['zefir_id'].isin(transition_ids)].to_string(index=False))

# And the gap diagnostic
print("\nIn mask but not reg:", sorted(mask_ids - reg_ids))
print("\nNumeric gaps in mask sequence:")
mask_nums = sorted(int(re.search(r'\d+', x).group()) for x in mask_ids)
print([n for a, b in zip(mask_nums, mask_nums[1:]) if b - a > 1 for n in range(a+1, b)])

print("\nNumeric gaps in reg sequence:")
reg_nums = sorted(int(re.search(r'\d+', x).group()) for x in reg_ids)
print([n for a, b in zip(reg_nums, reg_nums[1:]) if b - a > 1 for n in range(a+1, b)])

In [ ]:
# This SHOULD be as similar as possible to the registration pipeline downsampling behavior in run_slice_registration_optimized_newReg_sdf_v2.py
# Behavior to match:
# 1) symmetric proportional padding (prop_pad on each side),
# 2) extra bottom/right padding so shape is divisible by the downsample factor,
# 3) block-sum downsampling.


# ##this is effectively identical to downsample_image in slice_registration_functions.py, so we can get rid of it
# def downsample_with_registration_padding(image, rescale, prop_pad=0.2, pad_value=0):
#     arr = np.asarray(image)
#     if arr.ndim != 2:
#         raise ValueError(f"Expected a 2D array, got shape {arr.shape}")

#     #breaks if not int. as it stands, were doing this when we define rescale, but this shouldnt change the value if we redo it here to be safe
#     factor = max(1, int(round(rescale)))

#     # Symmetric proportional padding
#     pad0 = int(np.ceil(arr.shape[0] * prop_pad))
#     pad1 = int(np.ceil(arr.shape[1] * prop_pad))
#     arr = np.pad(arr, ((pad0, pad0), (pad1, pad1)), mode="constant", constant_values=pad_value)

#     # Extra padding only on bottom/right to make dimensions divisible by factor. TODO: Is this needed?
#     extra0 = (-arr.shape[0]) % factor
#     extra1 = (-arr.shape[1]) % factor
#     if extra0 or extra1:
#         arr = np.pad(arr, ((0, extra0), (0, extra1)), mode="constant", constant_values=pad_value)

#     return block_reduce(arr, block_size=(factor, factor), func=np.sum)


#crop or pad, depending if image is too large (crop) or too small (pad). The reg pipeline does this when it picks one image in the stack to use as a template. We need to do something similar, but since we have the actual source image shape and the target shape (after downsampling+padding), we can just do it to that image size directly. I think this works but TODO: test on other images.
def center_crop_or_pad_2d(image, target_shape, pad_value=0):
    arr = np.asarray(image)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D array, got shape {arr.shape}")

    target_h, target_w = int(target_shape[0]), int(target_shape[1])

    # Center-crop if too large.
    start_h = max((arr.shape[0] - target_h) // 2, 0)
    start_w = max((arr.shape[1] - target_w) // 2, 0)
    end_h = start_h + min(target_h, arr.shape[0])
    end_w = start_w + min(target_w, arr.shape[1])
    arr = arr[start_h:end_h, start_w:end_w]

    # Center-pad if too small.
    pad_h_total = max(target_h - arr.shape[0], 0)
    pad_w_total = max(target_w - arr.shape[1], 0)
    pad_h0 = pad_h_total // 2
    pad_h1 = pad_h_total - pad_h0
    pad_w0 = pad_w_total // 2
    pad_w1 = pad_w_total - pad_w0

    if pad_h_total or pad_w_total:
        arr = np.pad(arr, ((pad_h0, pad_h1), (pad_w0, pad_w1)), mode="constant", constant_values=pad_value)

    return arr


In [ ]:
# Generate list of slice directories by globbing based on unique numeric prefix (zefir_XXXX_...)
# `glob` may be a function from previous imports, so call it directly.
orig_fnames = sorted(glob(os.path.join(root_dir, '*realID_*_zefirID_????_*_pix.nii.gz')))

max_errors = []
missing_slices = []
bad_slices = []
max_error_thresh = 5.0

for name_idx, orig_fname in enumerate(orig_fnames):
    slice_name = os.path.basename(orig_fname)
    base_slice = slice_name.split('_downsample_10p002um_pix')[0]

    # Extract the shared prefix (before _Image_) to match prefixed mask files too.
    fname_header = re.search(r'(realID_[\d-]+_zefirID_\d+)', base_slice).group(1)
    labeled_mask = glob(os.path.join(mask_dir, f"*{fname_header}_*_mask*.ome.tif"))

    if len(labeled_mask) == 0:
        # print(f"No labeled mask found for {slice_name}")
        continue
    elif len(labeled_mask) > 1:
        # print(f"Multiple labeled masks found for {slice_name}: {labeled_mask}")
        continue

    labeled_mask_fname = labeled_mask[0]

    source_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix.nii.gz")
    mapping_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg0nl_ants-map.nii.gz")
    mapping_img3 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_ants-map.nii.gz")
    mapping_img4 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter3_ants-map.nii.gz")
    mapping_img5 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter4_ants-map.nii.gz")
    mapping_img6 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter5_ants-map.nii.gz")
    mapping_img7 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter6_ants-map.nii.gz")
    mapping_img8 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter7_ants-map.nii.gz")
    mapping_img9 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter8_ants-map.nii.gz")
    mapping_img10 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter9_ants-map.nii.gz")

    final_comparison_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter9_ants-def0.nii.gz")

    if not (os.path.exists(source_img) and os.path.exists(mapping_img) and os.path.exists(mapping_img3) and os.path.exists(mapping_img4)):
        missing_slices.append(slice_name)
        continue

    print(slice_name)
    src = nb.load(source_img)
    m1_ = nb.load(mapping_img)
    m2_ = nb.load(mapping_img3)
    m3_ = nb.load(mapping_img4)
    m4_ = nb.load(mapping_img5)
    m5_ = nb.load(mapping_img6)
    m6_ = nb.load(mapping_img7)
    m7_ = nb.load(mapping_img8)
    m8_ = nb.load(mapping_img9)
    m9_ = nb.load(mapping_img10)

    labeled_mask = tiff.imread(labeled_mask_fname).astype(bool).astype(np.uint8)  # read and convert to bin

    # centroid_seed_img = build_centroid_image(labeled_mask)
    centroid_seed_img = labeled_mask  # skipping centroids bc it takes too long for testing
    _ds_label_cnt = downsample_image(centroid_seed_img, rescale, prop_pad=0)
    _ds_label_cnt = center_crop_or_pad_2d(_ds_label_cnt, target_shape=src.shape)
    # _ds_label_cnt = downsample_image(labeled_mask, rescale, pad_value=-1*0) #now counts per pixel

    ## this breaks because the downsampled label count image is not the same shape as the source image
    # _d = np.zeros_like(src.get_fdata())
    # _d[...] = _ds_label_cnt
    # label_cnt_img = nb.Nifti1Image(_d, affine=src.affine, header=src.header)

    label_cnt_img = nb.Nifti1Image(_ds_label_cnt, affine=src.affine, header=src.header)

    comparison = nb.load(final_comparison_img).get_fdata()
    # seq = apply_coordinate_mapping_2d(apply_coordinate_mapping_2d(apply_coordinate_mapping_2d(src, m1_), m2_), m3_).get_fdata()
    labeled_seq = apply_coordinate_mapping_2d(
        apply_coordinate_mapping_2d(
            apply_coordinate_mapping_2d(
                apply_coordinate_mapping_2d(
                    apply_coordinate_mapping_2d(
                        apply_coordinate_mapping_2d(
                            apply_coordinate_mapping_2d(
                                apply_coordinate_mapping_2d(
                                    apply_coordinate_mapping_2d(
                                        label_cnt_img, m1_), m2_), m3_), m4_), m5_), m6_), m7_), m8_), m9_)

    seq = apply_coordinate_mapping_2d(
        apply_coordinate_mapping_2d(
            apply_coordinate_mapping_2d(
                apply_coordinate_mapping_2d(
                    apply_coordinate_mapping_2d(
                        apply_coordinate_mapping_2d(
                            apply_coordinate_mapping_2d(
                                apply_coordinate_mapping_2d(
                                    apply_coordinate_mapping_2d(
                                        src, m1_), m2_), m3_), m4_), m5_), m6_), m7_), m8_), m9_).get_fdata()

    final_space_labels['img'].append(labeled_seq)
    final_space_labels['fname_header'].append(fname_header)
    final_space_labels['final_reg_space_def0'].append(nb.load(final_comparison_img))

    max_err = np.abs(seq - comparison).max()
    max_errors.append(max_err)
    if max_err > max_error_thresh:
        bad_slices.append((slice_name, max_err, name_idx))

max_errors = np.array(max_errors)
print(f"Processed {len(max_errors)} slices successfully.")
if missing_slices:
    print(f"Skipped {len(missing_slices)} slices due to missing files: {missing_slices[:10]}{'...' if len(missing_slices) > 10 else ''}")

print(f"Processed {len(max_errors)} slices successfully.")
if missing_slices:
    print(f"Skipped {len(missing_slices)} slices due to missing files: {missing_slices[:10]}{'...' if len(missing_slices) > 10 else ''}")

plt.figure(figsize=(10, 5))
plt.hist(max_errors, bins=20, color='tab:blue', edgecolor='black')
plt.title('Max error per slice: apply_mapping_chain_exact vs def0 (final comparison)')
plt.xlabel('Max error')
plt.ylabel('Number of slices')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(max_errors[max_errors<max_error_thresh], bins=20, color='tab:blue', edgecolor='black')
plt.title('Max error per slice: apply_mapping_chain_exact vs def0 (final comparison, missing slices removed)')

plt.xlabel('Max error')
plt.ylabel('Number of slices')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()




In [ ]:
# orig_fnames = sorted(glob(os.path.join(root_dir, 'zefir_????_*_pix.nii.gz')))
# #hardcode specific slice with large visual shift for testing
orig_fnames = sorted(glob(os.path.join(root_dir, '*realID_*_zefirID_????_*_pix.nii.gz')))

max_errors = []
missing_slices = []
bad_slices = [] #exceed max_error_thresh
max_error_thresh = 5.0
final_space_labels = {}
final_space_labels['img'] =[]
final_space_labels['fname_header'] =[]
final_space_labels['final_reg_space_def0'] = []
for name_idx, orig_fname in enumerate(orig_fnames):

    slice_name = os.path.basename(orig_fname)
    base_slice = slice_name.split('_downsample_10p002um_pix')[0]

    # Extract the shared prefix token even when filenames have leading tags.
    header_match = re.search(r'(realID_[\d-]+_zefirID_\d+)', base_slice)
    if header_match is None:
        continue
    fname_header = header_match.group(1)
    labeled_mask = glob(os.path.join(mask_dir, f"*{fname_header}_*_mask*.ome.tif"))

    if len(labeled_mask) == 0:
        # print(f"No labeled mask found for {slice_name}")
        continue
    elif len(labeled_mask) > 1:
        # print(f"Multiple labeled masks found for {slice_name}: {labeled_mask}")
        continue

    labeled_mask_fname = labeled_mask[0]
    if os.path.exists(labeled_mask_fname):
        print(f"Found labeled mask for {slice_name}: {labeled_mask_fname}")

    source_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix.nii.gz")
    mapping_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg0nl_ants-map.nii.gz")
    mapping_img3 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_ants-map.nii.gz")
    mapping_img4 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter3_ants-map.nii.gz")
    mapping_img5 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter4_ants-map.nii.gz")
    mapping_img6 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter5_ants-map.nii.gz")
    mapping_img7 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter6_ants-map.nii.gz")
    mapping_img8 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter7_ants-map.nii.gz")
    mapping_img9 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter8_ants-map.nii.gz")
    mapping_img10 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter9_ants-map.nii.gz")

    final_comparison_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter3_ants-def0.nii.gz")
    final_comparison_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter9_ants-def0.nii.gz")

    if not (os.path.exists(source_img) and os.path.exists(mapping_img) and os.path.exists(mapping_img3) and os.path.exists(mapping_img4)):
        missing_slices.append(slice_name)
        continue
    print(slice_name)
    src = nb.load(source_img)
    m1_ = nb.load(mapping_img)
    m2_ = nb.load(mapping_img3)
    m3_ = nb.load(mapping_img4)
    m4_ = nb.load(mapping_img5)
    m5_ = nb.load(mapping_img6)
    m6_ = nb.load(mapping_img7)
    m7_ = nb.load(mapping_img8)
    m8_ = nb.load(mapping_img9)
    m9_ = nb.load(mapping_img10)

    labeled_mask = tiff.imread(labeled_mask_fname).astype(bool).astype(np.uint8) # read and convert to bin

    # centroid_seed_img = build_centroid_image(labeled_mask)
    centroid_seed_img = labeled_mask #skipping centroids bc it takes too long for testing
    _ds_label_cnt = downsample_image(centroid_seed_img, rescale, prop_pad=0)
    _ds_label_cnt = center_crop_or_pad_2d(_ds_label_cnt, target_shape=src.shape)
    # _ds_label_cnt = downsample_image(labeled_mask, rescale, pad_value=-1*0) #now counts per pixel

    ## this breaks because the downsampled label count image is not the same shape as the source image
    # _d = np.zeros_like(src.get_fdata())
    # _d[...] = _ds_label_cnt
    # label_cnt_img = nb.Nifti1Image(_d, affine=src.affine, header=src.header)

    label_cnt_img = nb.Nifti1Image(_ds_label_cnt, affine=src.affine, header=src.header)



    # if src.shape != label_cnt_img.shape:
    #     raise ValueError(f"Shape mismatch betweem source image and label count image: {src.shape} vs {label_cnt_img.shape}")

    # #create sham data full of 0s with one value in the middle of the image to track how it moves across the transformations
    # sham_data = np.zeros(src.shape, dtype=np.float32)
    # label_cnt_img = nb.Nifti1Image(_ds_label_cnt, affine=src.affine, header=src.header)

    comparison = nb.load(final_comparison_img).get_fdata()
    # seq = apply_coordinate_mapping_2d(apply_coordinate_mapping_2d(apply_coordinate_mapping_2d(src, m1_), m2_), m3_).get_fdata()
    labeled_seq = apply_coordinate_mapping_2d(
        apply_coordinate_mapping_2d(
            apply_coordinate_mapping_2d(
                apply_coordinate_mapping_2d(
                    apply_coordinate_mapping_2d(
                        apply_coordinate_mapping_2d(
                            apply_coordinate_mapping_2d(
                                apply_coordinate_mapping_2d(
                                    apply_coordinate_mapping_2d(
                                        label_cnt_img, m1_), m2_), m3_), m4_), m5_), m6_), m7_), m8_), m9_)

    seq = apply_coordinate_mapping_2d(
        apply_coordinate_mapping_2d(
            apply_coordinate_mapping_2d(
                apply_coordinate_mapping_2d(
                    apply_coordinate_mapping_2d(
                        apply_coordinate_mapping_2d(
                            apply_coordinate_mapping_2d(
                                apply_coordinate_mapping_2d(
                                    apply_coordinate_mapping_2d(
                                        src, m1_), m2_), m3_), m4_), m5_), m6_), m7_), m8_), m9_).get_fdata()

    final_space_labels['img'].append(labeled_seq)
    final_space_labels['fname_header'].append(fname_header)
    final_space_labels['final_reg_space_def0'].append(nb.load(final_comparison_img))

    max_err = np.abs(seq -comparison).max()
    max_errors.append(max_err)
    if max_err > max_error_thresh:
        bad_slices.append((slice_name, max_err, name_idx))

max_errors = np.array(max_errors)
print(f"Processed {len(max_errors)} slices successfully.")
if missing_slices:
    print(f"Skipped {len(missing_slices)} slices due to missing files: {missing_slices[:10]}{'...' if len(missing_slices) > 10 else ''}")

print(f"Processed {len(max_errors)} slices successfully.")
if missing_slices:
    print(f"Skipped {len(missing_slices)} slices due to missing files: {missing_slices[:10]}{'...' if len(missing_slices) > 10 else ''}")

plt.figure(figsize=(10, 5))
plt.hist(max_errors, bins=20, color='tab:blue', edgecolor='black')
plt.title('Max error per slice: apply_mapping_chain_exact vs def0 (final comparison)')
plt.xlabel('Max error')
plt.ylabel('Number of slices')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(max_errors[max_errors<max_error_thresh], bins=20, color='tab:blue', edgecolor='black')
plt.title('Max error per slice: apply_mapping_chain_exact vs def0 (final comparison, missing slices removed)')

plt.xlabel('Max error')
plt.ylabel('Number of slices')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()




In [ ]:
#load csv and look at detection sizes:

df = pd.read_csv("/data/neuralabc/johmat/phase_ml/QuPath_projects/output_masks_proj/measurements.csv")


plt.figure(figsize=(10, 5))
plt.hist(df['Area µm^2'], bins=200, color='tab:blue', edgecolor='black')
plt.xlabel('Area (µm²)')
plt.ylabel('Count')
plt.title('Detection sizes')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

print(df['Area µm^2'].describe())

In [ ]:
from scipy import ndimage
import tifffile as tiff
import numpy as np
from pathlib import Path

def filter_by_size(mask: np.ndarray, min_area_px: int) -> np.ndarray:
    labeled, n = ndimage.label(mask > 0)
    sizes = ndimage.sum(mask > 0, labeled, range(1, n + 1))
    keep = np.zeros_like(mask)
    for i, size in enumerate(sizes, start=1):
        if size >= min_area_px:
            keep[labeled == i] = 255
    return keep.astype(np.uint8)


# --- configure ---
input_path  = Path("/data/neuralabc/johmat/phase_ml/processing/full_slide_masks/2026_05_14/2026_05_14_pct_hi99.5_lo95/2026_05_14_pct_hi99.5_lo95_realID_050-1_zefirID_0112_Image_49_mask_pyr_stable.ome.tif")
output_path = Path("/data/neuralabc/johmat/phase_ml/processing/full_slide_masks/2026_05_14/size_filtered/2026_05_14_pct_hi99.5_lo95_realID_050-1_zefirID_0112_Image_49_mask_pyr_stable.ome.tif")
pixel_size_um = 0.345
min_area_um2  = 150
min_area_px   = int(min_area_um2 / (pixel_size_um ** 2))
print(f"Min area: {min_area_px} pixels")  # ~1260 px
# -----------------

mask = tiff.imread(str(input_path))
filtered = filter_by_size(mask, min_area_px)

print(f"Before: {np.unique(*np.where(mask > 0), return_counts=False)} blobs")  # rough
before = ndimage.label(mask > 0)[1]
after  = ndimage.label(filtered > 0)[1]
print(f"Blobs before: {before}, after: {after}, removed: {before - after}")

tiff.imwrite(str(output_path), filtered)

In [ ]:
output_dir = Path("/data/neuralabc/johmat/phase_ml/processing/full_slide_masks/2026_05_14/registered_slices")
output_dir.mkdir(parents=True, exist_ok=True)

for fname_header, labeled_seq in zip(final_space_labels['fname_header'], final_space_labels['img']):
    out_path = output_dir / f"{fname_header}_mask_registered.nii.gz"
    nb.save(labeled_seq, str(out_path))
    print(f"Saved: {out_path.name}")

In [ ]:
# Stack all registered masks into a single 3D volume (slice per z)
stack = np.stack([img.get_fdata()[:, :] for img in final_space_labels['img']], axis=-1)

# Use affine/header from first image as reference
ref = final_space_labels['img'][0]
stack_nii = nb.Nifti1Image(stack, affine=ref.affine, header=ref.header)

out_path = output_dir / "registered_masks_stack.nii.gz"
nb.save(stack_nii, str(out_path))
print(f"Saved stack: {stack_nii.shape}")

In [ ]:
import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
import nibabel as nb

heatmap_dir = Path("/data/neuralabc/johmat/phase_ml/tmp/Positive")
output_dir  = Path("/data/neuralabc/johmat/phase_ml/tmp/Positive/heatmaps")
output_dir.mkdir(exist_ok=True)
sigma = 4
files = sorted([f for f in heatmap_dir.iterdir() if f.name.endswith('.tif') or f.name.endswith('.nii.gz')])

for imnumber, fpath in enumerate(files):
    if fpath.name.endswith('.nii.gz'):
        mask = nb.load(str(fpath)).get_fdata().astype(np.float32)
        if mask.ndim == 3:
            mask = mask[:, :, 0]
    else:
        mask = tiff.imread(str(fpath)).astype(np.float32)
    ksize = int(sigma * 6) | 1
    heatmap = cv2.GaussianBlur(mask, (ksize, ksize), sigmaX=sigma)

    plt.figure(figsize=(10, 8))
    plt.imshow(heatmap, cmap='viridis', interpolation='nearest', vmax=0.05)
    plt.axis('off')
    plt.savefig(output_dir / f"{fpath.stem}_heatmap.png", dpi=1200, bbox_inches='tight', pad_inches=0)
    plt.close()
    print(f"Saved {imnumber}: {fpath.name}")

In [ ]:
mask.shape

In [ ]:
idx = 12  # whichever slice you want
img_data = final_space_labels['img'][idx].get_fdata()
bg_data  = final_space_labels['final_reg_space_def0'][idx].get_fdata()

fig, axes = plt.subplots(1, 3, figsize=(12, 5))

axes[0].imshow(bg_data[:, :], cmap='gray')
axes[0].set_title(f"Registered slice: {final_space_labels['fname_header'][idx]}")

axes[1].imshow(bg_data[:, :], cmap='gray')
axes[1].imshow(img_data[:, :], cmap='hot', alpha=0.8)
axes[1].set_title('With mask overlay')

axes[2].imshow(img_data[:, :], cmap='hot')
axes[2].set_title('Mask only')

plt.tight_layout()
plt.show()

In [ ]:
#histogram of values in the downsampled images

img_dir = "/tmp/zefir_sliceReg_optimized_v2_rescale_5_applied_mappings_output"
img_fnames = sorted(glob(os.path.join(img_dir, '*.nii.gz')))
all_img_data = []
for img_fname in img_fnames:
    img = nb.load(img_fname).get_fdata()
    img_data = img.flatten()
    all_img_data.append(img_data)

all_img_data = np.concatenate(all_img_data)
print(f"Max: {all_img_data.max()}")
print(f"Mean: {all_img_data.mean()}")
print(f"Non-zero Mean: {all_img_data[all_img_data > 0].mean()}")
print(f"Median: {np.median(all_img_data)}")
print(f"Non-zero Median: {np.median(all_img_data[all_img_data > 0])}")

plt.figure(figsize=(10, 5))
plt.hist(all_img_data, bins=50, color='tab:blue', edgecolor='black')
plt.title('Histogram of values in all registered images')
plt.show()

In [ ]:
# --- Configure these ---
csv_path = "/data/neuralabc/johmat/microscopy_scripts/macaque_CB/all_TP_image_idxs_file_lookup_UPDATED.csv"
dry_run  = False
# -----------------------

import re
import pandas as pd
from pathlib import Path


def extract_image_label(fname: str) -> str:
    m = re.search(r'Image_(\d+).*?_20x_(\d+)', fname)
    if m:
        return f"Image_{m.group(1)}_{m.group(2)}"
    m = re.search(r'Image_(\d+).*?_20x', fname)
    if m:
        return f"Image_{m.group(1)}"
    raise ValueError(f"Could not extract image numbers from: {fname}")


def build_output_name(idx: int, image_idx: float, johmat_file_name: str) -> str:
    major, minor = str(image_idx).split('.')
    image_idx_str = f"{int(major):03d}-{minor}"
    zefir_str = f"{int(idx):04d}"
    image_label = extract_image_label(johmat_file_name)
    return f"realID_{image_idx_str}_zefirID_{zefir_str}_{image_label}.ome.tif"


df = pd.read_csv(csv_path)

skipped, errors, processed = [], [], 0

for _, row in df.iterrows():
    idx = int(row['idx'])
    image_idx = row['image_idx']
    johmat_file_name = row['johmat_file_name']
    johmat_project_root = row['johmat_project_root']

    if pd.isna(johmat_file_name) or pd.isna(johmat_project_root):
        skipped.append(idx)
        continue

    try:
        src = Path(str(johmat_project_root).rstrip('/')) / str(johmat_file_name)
        out_name = build_output_name(idx, image_idx, str(johmat_file_name))
        dst = src.parent / out_name

        if dry_run:
            print(f"[DRY RUN] idx={idx:>3}  {src.name}")
            print(f"               -> {dst.name}")
        else:
            if not src.exists():
                raise FileNotFoundError(f"Source not found: {src}")
            src.rename(dst)
            print(f"Renamed: {src.name} -> {dst.name}")

        processed += 1

    except Exception as e:
        errors.append((idx, str(e)))

print(f"\n--- Summary ---")
print(f"Processed : {processed}")
print(f"Skipped (no johmat path): {len(skipped)}")
if skipped:
    print(f"  idx values: {skipped}")
print(f"Errors    : {len(errors)}")
for idx, msg in errors:
    print(f"  idx={idx}: {msg}")

In [ ]:
# --- Configure these ---
csv_path   = "/data/neuralabc/johmat/microscopy_scripts/macaque_CB/all_TP_image_idxs_file_lookup_UPDATED.csv"
masks_dir  = "/tmp/zefir_sliceReg_optimized_v2_rescale_5_renamed_copies/"
dry_run    = False
# -----------------------

import re
import pandas as pd
from pathlib import Path


def extract_image_label(fname: str) -> str:
    m = re.search(r'Image_(\d+).*?_20x_(\d+)', fname)
    if m:
        return f"Image_{m.group(1)}_{m.group(2)}"
    m = re.search(r'Image_(\d+).*?_20x', fname)
    if m:
        return f"Image_{m.group(1)}"
    raise ValueError(f"Could not extract image numbers from: {fname}")

def extract_suffix(fname: str) -> str:
    m = re.search(r'Image_\d+.*?_20x(?:_\d+)?(.*)', fname)
    if m:
        return m.group(1)
    raise ValueError(f"Could not extract suffix from: {fname}")


def build_output_name(idx: int, image_idx: float, fname: str) -> str:
    major, minor = str(image_idx).split('.')
    image_idx_str = f"{int(major):03d}-{minor}"
    zefir_str = f"{int(idx):04d}"
    image_label = extract_image_label(fname)
    suffix = extract_suffix(fname)
    return f"realID_{image_idx_str}_zefirID_{zefir_str}_{image_label}{suffix}"

df = pd.read_csv(csv_path)
idx_lookup = {int(row['idx']): row for _, row in df.iterrows()}

mask_files = list(Path(masks_dir).glob("*.nii.gz"))
skipped, errors, processed = [], [], 0

for src in mask_files:
    m = re.search(r'zefir_(\d{4})', src.name)
    if not m:
        skipped.append(src.name)
        continue

    zefir_idx = int(m.group(1))

    if zefir_idx not in idx_lookup:
        skipped.append(src.name)
        print(f"No CSV entry for zefir idx {zefir_idx}: {src.name}")
        continue

    try:
        row = idx_lookup[zefir_idx]
        out_name = build_output_name(zefir_idx, row['image_idx'], src.name)
        dst = src.parent / out_name

        if dry_run:
            print(f"[DRY RUN] {src.name}")
            print(f"       -> {dst.name}")
        else:
            src.rename(dst)
            print(f"Renamed: {src.name} -> {dst.name}")

        processed += 1

    except Exception as e:
        errors.append((src.name, str(e)))

print(f"\n--- Summary ---")
print(f"Processed : {processed}")
print(f"Skipped   : {len(skipped)}")
print(f"Errors    : {len(errors)}")
for name, msg in errors:
    print(f"  {name}: {msg}")

In [ ]:
import nibabel as nib
import numpy as np
import glob
import os
import re

directory = "/tmp/zefir_sliceReg_optimized_v2_rescale_5_applied_mappings_output"

def sort_key(f):
    m = re.search(r'realID_(\d{3})-(\d)', f)
    idx = int(m.group(1))
    sub = int(m.group(2))
    return (idx, sub)

for f in glob.glob(os.path.join(directory, "*.nii.gz")):
    m = re.search(r'realID_(\d{3})-(\d)', f)
    if not m:
        print(f"NO MATCH: {os.path.basename(f)}")

all_files = glob.glob(os.path.join(directory, "*.nii.gz"))
files = sorted(
    [f for f in all_files if re.search(r'realID_(\d{3})-(\d)', f)],
    key=sort_key
)
print(f"Found {len(all_files)} files, stacking {len(files)}")


volume = np.stack([nib.load(f).get_fdata() for f in files], axis=-1)
print(f"Stacked {len(files)} files -> shape {volume.shape}")

ref = nib.load(files[0])
out = nib.Nifti1Image(volume, affine=ref.affine, header=ref.header)
nib.save(out, os.path.join(directory, "stacked_volume.nii.gz"))

In [ ]:
import numpy as np
import pandas as pd
import tifffile
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from pathlib import Path

import os
import re

# ── CONFIG ────────────────────────────────────────────────────────────────────
POINTS_CSV  = "/data/neuralabc/johmat/phase_ml/QuPath_projects/Validation_Set_MGM/measurements.csv"
SQUARES_CSV = "/data/neuralabc/johmat/phase_ml/QuPath_projects/Validation_Set_MGM/measurements_annotations.csv"
# MASK_DIR is overridable from the environment so a parameter sweep can point at
# each setting's output dir without editing this file:
#   MASK_DIR=/data/.../sweep/<run> python eval_pr_tolerance.py
MASK_DIR = os.environ.get(
    "MASK_DIR",
    "/data/neuralabc/johmat/phase_ml/processing/full_slide_masks/validation/v3",
)

UM_PER_PX   = 0.3449
N_WORKERS   = min(os.cpu_count(), 8)  # tune to your machine

# Matching tolerance: a GT point and a blob centroid are the same cell if their
# centroids are within this distance. ~one Purkinje soma radius. Tune against the
# reported match-distance stats: if the median match distance is much smaller you
# can tighten; if real cells are being missed at the boundary, loosen.
MATCH_TOL_UM = 15.0
MATCH_TOL_PX = MATCH_TOL_UM / UM_PER_PX

POINTS_IMG_COL  = "Image"
X_COL           = "Centroid X µm"
Y_COL           = "Centroid Y µm"

SQUARES_IMG_COL = "Image"
CX_COL          = "Centroid X µm"
CY_COL          = "Centroid Y µm"
AREA_COL        = "Area µm^2"
# ─────────────────────────────────────────────────────────────────────────────


def find_mask_for_image(img_name, mask_dir):
    m = re.search(r'zefir_(\d+)', img_name)
    if not m:
        return None
    zefir_id = m.group(1).zfill(4)
    candidates = list(mask_dir.glob(f"*zefirID_{zefir_id}*.tif"))
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        print(f"  WARNING: {len(candidates)} candidates for {img_name}, using: {candidates[0].name}")
        return candidates[0]
    return None


def um_to_px(val):
    return val / UM_PER_PX


def prf(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


def match_points_to_blobs(pts_xy, blob_xy, tol_px):
    """
    One-to-one matching between GT points and blob centroids by nearest distance
    under a tolerance, via optimal (Hungarian) assignment.

    pts_xy  : (Np, 2) GT point coords (x, y) in mask pixels
    blob_xy : (Nb, 2) blob centroid coords (x, y) in mask pixels
    tol_px  : max centroid distance for a valid match

    Returns (tp, matched_dists, fn, fp):
      tp            : number of matched pairs (== matched points == matched blobs)
      matched_dists : list of the matched-pair distances (px), for diagnostics
      fn            : GT points with no match (misses, incl. one side of a merge)
      fp            : blobs with no match (spurious / over-segmentation fragments)

    A merged blob (two GT, one blob) yields 1 TP + 1 FN because the second GT
    cannot reuse the already-matched blob. An over-segmented cell (one GT, two
    blobs) yields 1 TP + 1 FP. Each error is counted once, not doubled.
    """
    Np = len(pts_xy)
    Nb = len(blob_xy)
    if Np == 0 or Nb == 0:
        return 0, [], Np, Nb

    # Pairwise Euclidean distances (Np, Nb).
    d = np.sqrt(((pts_xy[:, None, :] - blob_xy[None, :, :]) ** 2).sum(-1))

    # Forbid out-of-tolerance pairs with a large cost; they may still be returned
    # when the matrix is non-square, so we filter by true distance afterwards.
    BIG = 1e9
    cost = np.where(d <= tol_px, d, BIG)
    ri, ci = linear_sum_assignment(cost)

    matched_dists = []
    for r, c in zip(ri, ci):
        if d[r, c] <= tol_px:
            matched_dists.append(float(d[r, c]))

    tp = len(matched_dists)
    fn = Np - tp
    fp = Nb - tp
    return tp, matched_dists, fn, fp


def evaluate_image(args):
    """Top-level function required for multiprocessing pickling."""
    img_name, img_points_records, img_squares_records, mask_path_str = args

    img_points  = pd.DataFrame(img_points_records)
    img_squares = pd.DataFrame(img_squares_records)
    mask_path   = Path(mask_path_str)

    try:
        mask_raw = tifffile.imread(mask_path)
        if mask_raw.ndim > 2:
            mask_raw = mask_raw[0]
        mask = mask_raw > 0
    except Exception as e:
        return img_name, None, None, None, [], [], f"Failed to load mask: {e}"

    labeled, _ = ndimage.label(mask)

    blob_ids = np.unique(labeled)
    blob_ids = blob_ids[blob_ids > 0]

    if len(blob_ids) == 0:
        # No blobs at all -- every GT point is a miss.
        total_fn = len(img_points)
        return img_name, 0, 0, total_fn, [], [], None

    blob_centroids = ndimage.center_of_mass(mask, labeled, blob_ids)
    blob_cy = np.array([c[0] for c in blob_centroids])
    blob_cx = np.array([c[1] for c in blob_centroids])

    img_points  = img_points.copy()
    img_squares = img_squares.copy()

    img_points["x_px"] = um_to_px(img_points[X_COL])
    img_points["y_px"] = um_to_px(img_points[Y_COL])

    half_side_um = np.sqrt(img_squares[AREA_COL]) / 2
    img_squares["x1_px"] = um_to_px(img_squares[CX_COL] - half_side_um)
    img_squares["x2_px"] = um_to_px(img_squares[CX_COL] + half_side_um)
    img_squares["y1_px"] = um_to_px(img_squares[CY_COL] - half_side_um)
    img_squares["y2_px"] = um_to_px(img_squares[CY_COL] + half_side_um)

    image_tp = image_fp = image_fn = 0
    rect_results = []
    img_match_dists = []

    for _, rect in img_squares.iterrows():
        x1, x2 = rect["x1_px"], rect["x2_px"]
        y1, y2 = rect["y1_px"], rect["y2_px"]

        # Blobs whose centroid is inside the rect.
        in_rect_blob = (
            (blob_cx >= x1) & (blob_cx < x2) &
            (blob_cy >= y1) & (blob_cy < y2)
        )
        blob_xy = np.column_stack([blob_cx[in_rect_blob], blob_cy[in_rect_blob]])

        # GT points inside the rect.
        pts = img_points[
            (img_points["x_px"] >= x1) & (img_points["x_px"] < x2) &
            (img_points["y_px"] >= y1) & (img_points["y_px"] < y2)
        ]
        pts_xy = pts[["x_px", "y_px"]].to_numpy()

        tp, mdist, fn, fp = match_points_to_blobs(pts_xy, blob_xy, MATCH_TOL_PX)

        image_tp += tp
        image_fp += fp
        image_fn += fn
        img_match_dists.extend(mdist)

        rect_results.append({
            "rect_cx_um": rect[CX_COL],
            "rect_cy_um": rect[CY_COL],
            "n_points": len(pts),
            "n_blobs": int(in_rect_blob.sum()),
            "tp": tp, "fp": fp, "fn": fn,
            "med_match_px": float(np.median(mdist)) if mdist else np.nan,
        })

    return img_name, image_tp, image_fp, image_fn, rect_results, img_match_dists, None


# ── LOAD & PREP ───────────────────────────────────────────────────────────────
points_df  = pd.read_csv(POINTS_CSV)
squares_df = pd.read_csv(SQUARES_CSV)
mask_dir   = Path(MASK_DIR)

print(f"MASK_DIR = {mask_dir}")
print(f"Match tolerance = {MATCH_TOL_UM} um ({MATCH_TOL_PX:.1f} px)\n")

images = sorted(set(points_df[POINTS_IMG_COL].unique()) | set(squares_df[SQUARES_IMG_COL].unique()))
mask_lookup = {img: find_mask_for_image(img, mask_dir) for img in images}
missing = [img for img, mask_path in mask_lookup.items() if mask_path is None]
if missing:
    print(f"WARNING: mask not found for {len(missing)} image(s): {missing}\n")
images = [img for img in images if mask_lookup[img] is not None]


# Package args as plain dicts/lists so they survive pickling
task_args = [
    (
        img,
        points_df[points_df[POINTS_IMG_COL] == img].to_dict("records"),
        squares_df[squares_df[SQUARES_IMG_COL] == img].to_dict("records"),
        str(mask_lookup[img]),
    )
    for img in images
]

# ── SERIAL EXECUTION ───────────────────────────────────────────────────────────
all_results = []
all_match_dists = []

for i, args in enumerate(task_args, 1):
    img_name = args[0]
    img_name, tp, fp, fn, rect_results, match_dists, error = evaluate_image(args)

    if error:
        print(f"[{i}/{len(images)}] ERROR {img_name}: {error}")
        continue

    prec, rec, f1 = prf(tp, fp, fn)
    med_d = np.median(match_dists) if match_dists else np.nan
    all_match_dists.extend(match_dists)
    print(f"[{i}/{len(images)}] {img_name}  TP={tp} FP={fp} FN={fn}  "
          f"P={prec:.2%} R={rec:.2%} F1={f1:.2%}  med_d={med_d:.1f}px")

    all_results.append({
        "image": img_name,
        "n_rects": len(rect_results),
        "tp": tp, "fp": fp, "fn": fn,
        "precision": prec, "recall": rec, "f1": f1,
        "med_match_px": med_d,
    })

# ── RESULTS ───────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results).set_index("image")

print("\n=== Per-image results ===")
print(results_df.to_string())

total_tp = results_df["tp"].sum()
total_fp = results_df["fp"].sum()
total_fn = results_df["fn"].sum()
prec, rec, f1 = prf(total_tp, total_fp, total_fn)

print(f"\n=== Aggregate (micro-average across {len(results_df)} images) ===")
print(f"  TP : {total_tp}   FP : {total_fp}   FN : {total_fn}")
print(f"  Precision : {prec:.2%}")
print(f"  Recall    : {rec:.2%}")
print(f"  F1        : {f1:.2%}")

# Match-distance diagnostics: a large or skewed distribution flags a systematic
# offset (e.g. UM_PER_PX not equal to the mask's true MPP) rather than a detector
# problem, and tells you whether MATCH_TOL_UM is set sensibly.
if all_match_dists:
    md = np.array(all_match_dists)
    print(f"\n=== Match-distance (px) over {len(md)} TP pairs ===")
    print(f"  median : {np.median(md):.2f}   mean : {md.mean():.2f}   "
          f"p90 : {np.percentile(md, 90):.2f}   max : {md.max():.2f}")
    print(f"  tolerance was {MATCH_TOL_PX:.1f} px; "
          f"{100.0 * (md > 0.8 * MATCH_TOL_PX).mean():.1f}% of matches sit beyond 0.8*tol")

In [ ]:
import re
from pathlib import Path

SOURCE = Path("/data/neuralabc/johmat/phase_ml/source/macaque_tiffs")
MASKS  = Path("/data/neuralabc/johmat/phase_ml/processing/full_slide_masks/2026_06_01_v3/2026_06_08_pct_hi98.5_lo85")

ID_RE = re.compile(r"zefir(?:ID)?_(\d+)", re.IGNORECASE)

def extract(p):
    m = ID_RE.search(p.name)
    return m.group(1) if m else None

src   = {extract(f) for f in SOURCE.glob("*.ome.tif")} - {None}
masks = {extract(f) for f in MASKS.glob("*.ome.tif")}  - {None}

only_src   = sorted(src - masks)
only_masks = sorted(masks - src)
both       = sorted(src & masks)

print(f"Source slides : {len(src)}")
print(f"Mask outputs  : {len(masks)}")
print(f"In both       : {len(both)}")
print()
if only_src:
    print(f"In SOURCE but no mask ({len(only_src)}):")
    for x in only_src: print(f"  {x}")
else:
    print("In SOURCE but no mask: none")
print()
if only_masks:
    print(f"In MASKS but no source ({len(only_masks)}):")
    for x in only_masks: print(f"  {x}")
else:
    print("In MASKS but no source: none")

print(f"\nPaste this: {set(only_src)}")

In [ ]:
import matplotlib.pyplot as plt
import tifffile as tiff

img = tiff.imread("/data/neuralabc/johmat/phase_ml/source/macaque_ROI_masks_10um/Granule_Purkinje_layer/zefir_0115__Image_02_-_20x_Granule_Purkinje_layer.ome.tif")
plt.figure(figsize=(8, 8))
plt.imshow(img, cmap='gray')
plt.show()